# 05 — Scorecard

In [ ]:
# Cell 1 — setup
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent / "src"))
from scorelab import config, data
con = data.connect()            # read-only DuckDB connection to data/lc.duckdb

In [ ]:
# Cell 2 — sanity check
print(con.sql("SHOW TABLES"))
print(con.sql("SELECT COUNT(*) AS n, MIN(issue_d), MAX(issue_d) FROM accepted"))

In [ ]:
# Cell 3 — common imports and settings for this notebook
import json, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
from scorelab import target, attributes as A, binning as B, scorecard as S, validation as V, fairness as F
from scorelab import challengers as C, reject_inference as R, monitoring as M, report as REP
MODELS = config.MODELS_DIR; MODELS.mkdir(exist_ok=True)
SAMPLE_FRAC = None     # None = full data (recommended for the fitting notebooks); e.g. 0.10 for a quick pass
cols = [r[0] for r in con.execute("DESCRIBE v_target").fetchall()]
assert "label" in cols, "v_target is the old stub: close this kernel and run `python scripts/refresh_views.py`"
def load_frame(splits=("train", "validate", "oot"), **kw):
    """Observable, non-excluded loans with attributes built; indeterminates dropped unless asked."""
    f = target.modeling_frame(con, splits=splits, sample_frac=SAMPLE_FRAC, **kw)
    return A.build_attributes(f)
MEANINGS = {a.name: a.meaning for a in A.REGISTRY}
DIRECTIONS = {a.name: a.direction for a in A.REGISTRY}

## What this notebook settles

The champion card: which attributes enter, the coefficients, the points, the odds-to-score table and the reason codes. The fit is a logistic regression on WOE; the sign of every coefficient must be negative (higher WOE = safer), and the scaling follows `config.BASE_SCORE / BASE_ODDS / PDO`.

Decisions: **I — selection rules**, **J — forced inclusions and exclusions**, **K — penalty**, then the champion is saved to `models/champion/`.

In [ ]:
frame = load_frame()
tr, va, oot = (frame[frame.split == s] for s in ("train", "validate", "oot"))
specs = B.load_specs(MODELS / "bins.json"); binned = json.loads((MODELS / "binning_decisions.json").read_text(encoding="utf-8"))["binned_attributes"]
ivs = B.iv_summary(specs); order = ivs.attribute.tolist()
W = {k: B.transform_woe(d, specs) for k, d in (("train", tr), ("validate", va), ("oot", oot))}
print("train", tr.shape, "attributes available", len(order))

### Decision I — Selection rules

Forward selection in IV order with a p-value to enter, a correlation guard between WOE columns, and a maximum number of attributes. Fewer attributes mean a more readable card and more stable points; more attributes mean more Gini until it stops.

**Your reasoning:**

_(write here)_

In [ ]:
SEL = dict(p_enter=0.01, max_corr=0.70, max_attributes=15)
selected, log = S.stepwise_select(W["train"], tr["bad"], order, **SEL)
display(log); print("selected:", selected)

### Decision J — Forced in / out

An attribute the business expects on the card (delinquency history, inquiries) can be forced in even if the stepwise skipped it for correlation; one you cannot explain to a consumer is forced out.

**Your reasoning:**

_(write here)_

In [ ]:
FORCE_IN, FORCE_OUT = [], []
final_attrs = [a for a in selected if a not in FORCE_OUT] + [a for a in FORCE_IN if a not in selected]
card = S.fit_logistic(W["train"], tr["bad"], final_attrs)
display(S.coefficient_table(card)); display(S.vif_table(W["train"], final_attrs))
print(f"attributes {len(final_attrs)} · deviance {card.deviance:.0f} · null {card.null_deviance:.0f} · McFadden R2 {card.pseudo_r2:.4f}")

### Decision K — Penalty

Needed only if a coefficient is unstable or a bin was separated; an L2 penalty is a Gaussian prior on the coefficients and removes the standard errors from the fit. Default none.

**Your reasoning:**

_(write here)_

In [ ]:
PENALTY = None                          # e.g. 1.0 for ridge
if PENALTY:
    card = S.fit_logistic(W["train"], tr["bad"], final_attrs, penalty=PENALTY); display(S.coefficient_table(card))
pts = S.points_table(card, specs); display(pts)
pts.to_csv(config.DOCS_DIR / "exhibits" / "points_table.csv", index=False)

In [ ]:
# Evidence — performance at each level, before any calibration work
rows = []
scored = {}
for name, d in (("in-sample (train)", tr), ("out-of-sample (validate)", va), ("out-of-time (oot)", oot)):
    sc = S.score(d, card, specs); scored[name] = sc
    m = V.delong_ci(d["bad"], sc["score"]); f = V.auc_gini_ks(d["bad"], d["fico_mid"])
    rows.append({"level": name, "n": len(d), "bad_rate": round(d["bad"].mean(), 4), "gini_card": round(m["gini"], 4),
                 "gini_ci": tuple(round(x, 4) for x in m["gini_ci"]), "ks_card": round(V.auc_gini_ks(d["bad"], sc["score"])["ks"], 4),
                 "gini_fico": round(f["gini"], 4), "score_min": int(sc.score.min()), "score_max": int(sc.score.max())})
display(pd.DataFrame(rows))
sc = scored["out-of-time (oot)"]
display(S.odds_table(sc["score"], oot["bad"], card, band=20))
display(S.reason_codes(oot.head(5), card, specs, meanings=MEANINGS).T)

In [ ]:
# Save the champion
S.save_card(card, specs, MODELS / "champion")
(MODELS / "champion" / "decisions.json").write_text(json.dumps({"attributes": final_attrs, "selection": SEL, "force_in": FORCE_IN, "force_out": FORCE_OUT, "penalty": PENALTY}, indent=1), encoding="utf-8")
print("saved", MODELS / "champion")